In [11]:
# importamos las librerías que necesitamos

# Tratamiento de datos
# -----------------------------------------------------------------------
import pandas as pd
import numpy as np
from src import soporte_datasets as sp_datasets

# Visualización
# -----------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# Imputación de nulos usando métodos avanzados estadísticos
# -----------------------------------------------------------------------
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer


# Configuración
# -----------------------------------------------------------------------
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

# Gestión de los warnings
# -----------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

In [12]:
df_flight = pd.read_csv('files/Customer Flight Activity.csv') 
df_flight.head(2)

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0


In [13]:
df_loyalty = pd.read_csv('files/Customer Loyalty History.csv') #,index_col=0
df_loyalty.head(2)

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN


Lo primero a realizar es poner todos los datos en minúsculas, y en las columnas, cambiar los espacios por guiones para un mejor tratamiento de los datos. 

In [14]:
df_flight.columns = sp_datasets.min_datos(df_flight)

In [15]:
df_flight.head()

,loyalty_number,year,month,flights_booked,flights_with_companions,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0
2,100140,2017,1,6,0,6,1200,120.0,0,0
3,100214,2017,1,0,0,0,0,0.0,0,0
4,100272,2017,1,0,0,0,0,0.0,0,0


In [16]:
df_loyalty.columns = sp_datasets.min_datos(df_loyalty)

In [17]:
df_loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,2018.0,1.0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,NaN,NaN
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,NaN,NaN


# Gestión de valores nulos

Tal como hemos indicado en la fase inicial, existen nulos en las columnas de Salario y cancelación, vamos a tratarlos antes de la unión.

    - Para las columnas de cancelación de df_loyalty, vamos a pasar los Nulos como False, manteniendo el año y mes, ya que los valores nulos significan que el usuario sigue estando dado de alta en el sistema.

    - Para la columna Salario de df_loyalty, analizaremos patrones para ver la mejor opción.

In [19]:
df_loyalty["cancellation_year"]= sp_datasets.nulos_false_int(df_loyalty["cancellation_year"])
df_loyalty["cancellation_month"]= sp_datasets.nulos_false_int(df_loyalty["cancellation_month"])

In [20]:
df_loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,False,False
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,False,False
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,2018,1
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,False,False
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,False,False


In [34]:
(df_loyalty.isna().sum()/df_loyalty.shape[0]*100).sort_values(ascending=False).head(2)

#Comprobamos que solo está la columna Salario.

salary            25.321145
loyalty_number     0.000000
dtype: float64

In [38]:
df_loyalty.info()

#comprobamos que las columnas que acabamos de procesar es tipo objeto, tienen valores numericos y 
# boleanos, de momento vamos a dejarlo así por si tenemos que analizar % por mes.


<class 'pandas.DataFrame'>
RangeIndex: 16737 entries, 0 to 16736
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   loyalty_number      16737 non-null  int64  
 1   country             16737 non-null  str    
 2   province            16737 non-null  str    
 3   city                16737 non-null  str    
 4   postal_code         16737 non-null  str    
 5   gender              16737 non-null  str    
 6   education           16737 non-null  str    
 7   salary              12499 non-null  float64
 8   marital_status      16737 non-null  str    
 9   loyalty_card        16737 non-null  str    
 10  clv                 16737 non-null  float64
 11  enrollment_type     16737 non-null  str    
 12  enrollment_year     16737 non-null  int64  
 13  enrollment_month    16737 non-null  int64  
 14  cancellation_year   16737 non-null  object 
 15  cancellation_month  16737 non-null  object 
dtypes: float64(2), 

Para el salario, vamos a ver los salarios nulos con la Educación para ver si tienen patrón.

In [47]:
salario_negativo = df_loyalty[df_loyalty["salary"]<0]

In [53]:
salario_negativo["salary"].unique()
#Analizamos los salarios negativos.

array([-49830., -12497., -46683., -45962., -19325., -43234., -10605.,
       -17534., -58486., -31911., -49001., -34079.,  -9081., -46470.,
       -26322., -47310., -39503., -19332., -46303., -57297.])

In [63]:
len(salario_negativo["salary"].unique())
#Analizamos cuántos son

20

In [ ]:
salario_negativo['education'].value_counts()

#los datos corresponden a la mayoría de educación Bachelor.

education
Bachelor                19
High School or Below     1
Name: count, dtype: int64

In [71]:
salario_negativo[["education","salary"]].sort_values("salary")

,education,salary
7373,Bachelor,-58486.0
16735,Bachelor,-57297.0
1082,High School or Below,-49830.0
8767,Bachelor,-49001.0
14327,Bachelor,-47310.0
2471,Bachelor,-46683.0
12596,Bachelor,-46470.0
16431,Bachelor,-46303.0
3575,Bachelor,-45962.0
4712,Bachelor,-43234.0


Ahora que tengo los salarios negativos, voy a verlos junto con la educación

In [ ]:
media_salarios_limpios = df_loyalty[df_loyalty['salary'] > 0].groupby('education')['salary'].min().reset_index()
media_salarios_limpios.head()
#saco el valor mínimo de los salarios por nivel educativo.

,education,salary
0,Bachelor,15609.0
1,Doctor,48109.0
2,High School or Below,21853.0
3,Master,56414.0


Apreciamos que el valor minimo de Bachelor es 15609, y el valor negativo más bajo es 9081, seguido de 10605, interpretamos que la imputación de los datos ha habido una errata en el signo y los volvemos positivos, teniendo en cuenta que el valor de 9081 sea un posible outlier.

In [81]:
# Aplicamos el valor absoluto a la columna salary
df_loyalty['salary'] = df_loyalty['salary'].abs()

# Verificamos que ya no hay negativos
df_loyalty['salary'].min()

np.float64(9081.0)